# Webskraping: Versameling van mededingersdata

**NGRDA150 - Leereenheid 2.2: Datainsamelingsmetodes**

---

## Leereenheid 2.2: Eksterne databronne

### Agtergrond: Boland Meubels se uitdaging

Pieter van der Merwe (eienaar van Boland Meubels) het opgemerk dat sy winsmarges oor die afgelope 18 maande afgeneem het van 35% tot 28%. Een moontlike rede is dat mededingers meer kompeterende pryse aanbied.

Om hierdie hipotese te toets, moet Pieter eers verstaan watter pryse sy mededingers vir soortgelyke produkte vra. In hierdie praktikum leer jy hoe om produkpryse vanaf 'n mededinger se webwerf **outomaties** te versamel.

---

## Wat is webskraping?

**Webskraping** is die proses om data outomaties vanaf webwerwe te onttrek. In plaas daarvan om handmatig deur 'n webwerf te blaai en pryse in 'n Excel-sigblad te tik, skryf ons 'n Python-program wat die werk vir ons doen.

### Praktiese Toepassings in Besigheid:
- **Mededingingsanalise:** Versamel pryse van mededingers
- **Marknavorsing:** Ontleed verbruikersresensies
- **Voorraadbestuur:** Monitor produkbeskikbaarheid
- **Ekonomiese data:** Versamel wisselkoerse, aandelekoerse, ens.

---

## Etiese en wettige oorwegings

⚠️ **Belangrik:** Nie alle webwerwe laat web scraping toe nie!

Voordat jy enige webwerf skraap, moet jy:

1. **Nagaan of dit toegelaat is:** Lees die webwerf se `robots.txt` lêer
2. **Respekteer beperkings:** Moenie die bediener oorlaai nie
3. **Lees gebruiksvoorwaardes:** Sommige webwerwe verbied skraping in hul terme
4. **Oorweeg API's:** Baie webwerwe bied 'n amptelike API vir data-toegang



### Voorbeeld: Nagaan van robots.txt

Voeg `/robots.txt` by die einde van enige webwerf se URL:
- https://furniture-warehouse.co.za/robots.txt

In [ ]:
# Invoer van 'n biblioteek uit Python se standaardpakket
# wat gebruik word om webbladsye af te laai
import urllib.request

# Die adres van die robots.txt-lêer wat ons wil lees
url = "https://furniture-warehouse.co.za/robots.txt"

# Skep 'n webversoek na die robots.txt-lêer.
# Die 'User-Agent' identifiseer die tipe program wat die versoek stuur.
# Baie webbedieners verwag dat hierdie inligting ingesluit moet wees.
req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})

# Stuur die versoek na die webwerf en ontvang die antwoord
response = urllib.request.urlopen(req)

# Lees die inhoud van die antwoord.
# Die inhoud kom as 'bytes' terug, daarom word dit na gewone teks omgeskakel met decode().
inhoud = response.read().decode()

# Druk die inhoud van die robots.txt-lêer op die skerm
print(inhoud)

## Biblioteek gebruik
Ons gaan drie Python-biblioteke gebruik:

requests: Om HTML-inhoud vanaf 'n webwerf te haal
BeautifulSoup: Om die HTML te ontleed en data te onttrek
pandas: Om die data te stoor

## Stap 1: Invoer van biblioteke

In [ ]:
# Invoer van nodige biblioteke
import requests
from bs4 import BeautifulSoup
import pandas as pd

print("Biblioteke suksesvol ingevoer!")

## Stap 2: Haal HTML-inhoud vanaf 'n webwerf

Ons gaan die **Furniture Warehouse** webwerf gebruik as voorbeeld van 'n mededinger. Ons sal eetkamerstoele se pryse versamel.

In [ ]:
# Definieer die URL van die mededinger se produkbladsy
url = "https://furniture-warehouse.co.za/product-category/3-dining-room-furniture/dining-chairs/"

# Stuur 'n HTTP GET-versoek na die webwerf
response = requests.get(url)

# Kontroleer of die versoek suksesvol was
if response.status_code == 200:
    print(f"✓ Suksesvol gekonnekteer! Status kode: {response.status_code}")
    print(f"✓ HTML-grootte: {len(response.text)} karakters")
else:
    print(f"✗ Fout: Status kode {response.status_code}")

### Verduideliking:

- **Status kode 200:** Sukses! Die bediener het ons versoek beantwoord
- **Status kode 404:** Bladsy nie gevind nie
- **Status kode 403:** Toegang geweier (dalk blokkeer die webwerf scraping)

---

## Stap 3: Ontleed die HTML met BeautifulSoup

Die HTML wat ons ontvang het is net rou teks. BeautifulSoup help ons om dit te struktureer en spesifieke elemente te vind.

In [ ]:
# Skep 'n BeautifulSoup-objek
soup = BeautifulSoup(response.text, 'html.parser')

# Wys die bladsy se titel
print("Bladsy titel:")
print(soup.title.string)
print()

# Ons soek alle 'div' elemente wat die klas 'e-loop-item' het.
# Dit is die "houers" vir elke produk op die blad.
# Let wel: Die HTML-struktuur kan wissel per webwerf
produk_lys = soup.find_all('div', class_='e-loop-item')

# Tel hoeveel produkte op die bladsy is (soek vir produk-elemente)
print(f"Aantal produkte gevind: {len(produk_lys)}")

## Stap 4: Onttrek produkdata

Nou gaan ons deur elke produk loop en die volgende inligting onttrek:
- Produknaam
- Prys


In [ ]:
# Skep leë lyste om data te stoor
produkname = []
pryse = []

# Loop deur elke produk
for produk in produk_lys:

    #print(f"produk = {produk}\n")

    # Vind die produknaam
    naam_element = produk.find('h4', class_='product_title')
    #print(f"naam_element = {naam_element}")

    if naam_element:
        produkname.append(naam_element.text.strip())
    else:
        produkname.append('Onbekend')

    # Vind die prys
    # Op hierdie blad is die huidige prys gewoonlik binne 'n <ins> (insert) tag vir uitverkopings
    prys_element = produk.find('ins')
    #print(f"prys_element = {prys_element}\n\n\n")

    # As daar nie 'n uitverkoping is nie, soek ons die gewone prys-klas
    if not prys_element :
        prys_element  = produk.find('span', class_='woocommerce-Price-amount')

    prys_teks = prys_element.text.strip() if prys_element else "Prys op aanvraag"
    pryse.append(float(prys_teks.replace("R", "").replace(" ", "")))

print(f"Data suksesvol onttrek vir {len(produkname)} produkte!")

## Stap 5: Skep 'n Pandas DataFrame

Nou plaas ons die data in 'n DataFrame sodat ons dit maklik kan analiseer.

In [ ]:
# Skep die DataFrame
df = pd.DataFrame({
    'Produk': produkname,
    'Prys (R)': pryse
})

# Voeg 'n kolom by vir die databron
df['Mededinger'] = 'Furniture Warehouse'

# Wys die eerste paar rye
print("Eerste 10 produkte:")
print(df.head(10))

## Stap 6: Basiese Data-analise

Laat ons nou 'n paar basiese statistieke bereken om Pieter se besluite te ondersteun.

In [ ]:
# Verwyder rye waar prys ontbreek
df_clean = df.dropna(subset=['Prys (R)'])

print("=" * 50)
print("MEDEDINGER PRYSANALISE: EETKAMERSTOELE")
print("=" * 50)
print()

# Bereken statistieke
print(f"Aantal produkte geanaliseer: {len(df_clean)}")
print(f"Gemiddelde prys: R{df_clean['Prys (R)'].mean():.2f}")
print(f"Laagste prys: R{df_clean['Prys (R)'].min():.2f}")
print(f"Hoogste prys: R{df_clean['Prys (R)'].max():.2f}")
print(f"Mediaan prys: R{df_clean['Prys (R)'].median():.2f}")
print()


## Stap 7: Stoor die Data vir Verdere Analise

In [ ]:
# Stoor na 'n CSV-lêer
df.to_csv('mededinger_pryse_eetkamerstoele.csv', index=False, encoding='utf-8-sig')
print("✓ Data gestoor as 'mededinger_pryse_eetkamerstoele.csv'")

# Jy kan hierdie lêer later in Excel oopmaak of met Python verder analiseer

---

## Toepassing op Boland Meubels

### Hoe kan Pieter hierdie inligting gebruik?

1. **Prysstrategie:** Vergelyk sy eie pryse met die mededinger se gemiddelde
2. **Produkkeuse:** Identifiseer watter produkte meer gewild is (gebaseer op beskikbaarheid)
3. **Winsmargeberekening:** Bereken of hy sy pryse kan verhoog sonder om mededingendheid te verloor
4. **Markposisionering:** Besluit of hy op prys kompeteer of op kwaliteit/diens

---

**Let wel:** Die HTML-struktuur van webwerwe kan enige tyd verander. As hierdie kode nie meer werk nie, moet jy die webwerf se nuwe HTML-struktuur ondersoek en die selectors aanpas.